# spectra - a linear-algebra-first deep learning framework

**Raya Sergieva**

This notebook is the research report accompanying the
[spectra](https://github.com/RayaSergieva/ml-research-pipeline) framework, a
deep learning library written from scratch in NumPy. Everything the library
does is derived here from linear algebra and matrix calculus before it is
used, and the trained networks are then taken apart with the library's own
spectral instruments.

## 1. Problem formulation

Deep learning frameworks are usually consumed as black boxes. The premise of
this project is that every component of a working framework - automatic
differentiation, initialization, optimization, the training dynamics
themselves - is a small number of linear-algebraic ideas, and that building
the whole stack from those ideas produces both a working artifact and real
understanding.

The project therefore has two intertwined goals.

**The framework.** A NumPy-only library with a `Tensor` type, reverse-mode
automatic differentiation, neural network modules, optimizers, and a test
suite whose core is finite-difference gradient checking.

**The research question.** How does a neural network reorganize itself
*spectrally* during training? Concretely, how do the singular value spectra
of the weight matrices evolve as a network learns, and can that be exploited -
for example, through initialization schemes that set the spectrum
deliberately?

Scope for this report - a multilayer perceptron trained on MNIST, tracked
with a spectral analyzer, plus the mathematical development of every
component used along the way. The framework itself supports more
(the repository road map documents the ongoing extensions).

### Why this matters

Automatic differentiation is the single algorithm underlying all of modern
deep learning, yet it is just the chain rule organized around a graph.
Initialization theory decides whether a deep network trains at all, and it is
a variance propagation argument. The spectrum of a weight matrix is a
complete, orientation-free description of what a layer does to its input
space. These are exactly the topics a linear algebra course equips one to
treat rigorously, and this project does so end to end.

## 2. Tensors and the anatomy of a differentiable program

### 2.1 The Tensor

A tensor here is simply an $n$-dimensional array of real numbers together
with bookkeeping for differentiation. Formally, a tensor of shape
$(d_1, \dots, d_k)$ is an element of $\mathbb{R}^{d_1 \times \cdots \times d_k}$.
The framework wraps `numpy.ndarray` and adds three fields - `requires_grad`
marking the tensor as a differentiation variable, `grad` holding the
accumulated gradient, and an internal reference to the operation that
produced it, which is how the computational graph is stored.

In [1]:
import numpy as np

from spectra import Tensor

x = Tensor([[1.0, 2.0], [3.0, 4.0]], requires_grad=True)
x

Tensor([[1., 2.],
 [3., 4.]], dtype=float64, requires_grad=True)

### 2.2 Functions as graphs

A numerical program is a composition of primitive operations. Writing
$L = f_3(f_2(f_1(x)))$, each intermediate value is a node and each primitive
an edge, giving a directed acyclic graph. In spectra the graph is built
implicitly - evaluating an expression records, on each result, which
operation produced it and from which inputs.

In [2]:
a = Tensor([2.0], requires_grad=True)
b = Tensor([3.0], requires_grad=True)
c = a * b + a  # two primitive nodes, recorded automatically
c._ctx, [p.data for p in c._ctx.parents]

(<spectra.ops.Add at 0x7f6d507ebd10>, [array([6.]), array([2.])])

## 3. Matrix calculus and reverse-mode differentiation

### 3.1 Derivatives of vector- and matrix-valued maps

For $f : \mathbb{R}^n \to \mathbb{R}^m$, the derivative at $x$ is the Jacobian

$$
J_f(x) \in \mathbb{R}^{m \times n}, \qquad
[J_f(x)]_{ij} = \frac{\partial f_i}{\partial x_j}(x),
$$

the unique linear map satisfying $f(x + h) = f(x) + J_f(x)\,h + o(\lVert h\rVert)$.
For a composition $L = g(f(x))$ with $g : \mathbb{R}^m \to \mathbb{R}$ the chain
rule reads

$$
\nabla_x L \;=\; J_f(x)^{\top} \, \nabla_{f(x)} g .
$$

This single line is the entire backward pass. Every operation must be able to
answer one question - given the gradient of the loss with respect to my
output, what is the gradient with respect to each of my inputs? That map,
$v \mapsto J^{\top} v$, is the **vector-Jacobian product** (VJP). Note what is
*not* required - the Jacobian itself is never materialized. For a layer with
input and output dimension $10^3$, the Jacobian has $10^6$ entries per sample,
while the VJP is computed with a handful of matrix products.

### 3.2 Why reverse mode

Consider a scalar loss $L : \mathbb{R}^n \to \mathbb{R}$ built from $k$
primitives. Two evaluation orders of the chain rule exist.

*Forward mode* propagates $\partial(\cdot)/\partial x_j$ alongside the
computation, one input direction at a time - $n$ passes for a full gradient.

*Reverse mode* propagates $\partial L/\partial(\cdot)$ backward from the
output, all inputs at once - one pass, at a cost proportional to the forward
evaluation.

Training a network means $n$ in the millions (every weight) and a scalar $L$.
Reverse mode wins by a factor of $n$. This asymmetry, not any deep principle,
is why backpropagation is *the* algorithm of deep learning.

### 3.3 The VJP rules used by spectra

Each rule below is the transpose-Jacobian action, derived once and tested by
finite differences in `tests/test_autograd.py`. Writing $G$ for the incoming
gradient $\partial L / \partial(\text{output})$,

| operation | forward | VJP w.r.t. inputs |
|---|---|---|
| add | $C = A + B$ | $G$, $\;G$ |
| multiply | $C = A \odot B$ | $G \odot B$, $\;G \odot A$ |
| matmul | $C = AB$ | $G B^{\top}$, $\;A^{\top} G$ |
| sum | $s = \sum_i a_i$ | broadcast of $G$ |
| ReLU | $c_i = \max(a_i, 0)$ | $G \odot \mathbf{1}[a > 0]$ |
| exp | $c_i = e^{a_i}$ | $G \odot e^{a}$ |
| log | $c_i = \ln a_i$ | $G \oslash a$ |

The matmul rule is worth deriving explicitly since it anchors everything
else. With $L$ scalar and $C = AB$,

$$
\frac{\partial L}{\partial A_{ij}}
= \sum_{k} \frac{\partial L}{\partial C_{ik}} \frac{\partial C_{ik}}{\partial A_{ij}}
= \sum_{k} G_{ik} B_{jk}
= (G B^{\top})_{ij},
$$

and symmetrically $\partial L/\partial B = A^{\top} G$. The derivative of a
matrix product is again a pair of matrix products, which is what makes the
backward pass as cheap as the forward pass.

### 3.4 Broadcasting and its adjoint

NumPy silently *broadcasts* operands of unequal shape, replicating a smaller
array along new or size-one axes. Broadcasting is a linear map (replication),
so the chain rule needs its adjoint, and the adjoint of replication is
summation over the replicated axes. Every elementwise VJP in spectra is
therefore followed by an `_unbroadcast` step summing the gradient back down
to the operand's shape. Getting this wrong is the classic autograd bug, and
it is exactly the kind of error finite-difference checking catches.

### 3.5 Fan-out and gradient accumulation

When one tensor feeds several operations, the multivariate chain rule sums
the contributions,

$$
\frac{\partial L}{\partial x}
= \sum_{p \,\in\, \text{paths}} \frac{\partial L}{\partial u_p}\frac{\partial u_p}{\partial x},
$$

implemented as `parent.grad += g` while the graph is walked in reverse
topological order. The walk itself is an iterative depth-first
post-ordering, so graph depth is limited by memory rather than the Python
recursion limit.

In [3]:
# The engine at work - d/da of (a*b + a) is b + 1, and d/db is a.
a = Tensor([2.0], requires_grad=True)
b = Tensor([3.0], requires_grad=True)
(a * b + a).sum().backward()
a.grad, b.grad

(array([4.]), array([2.]))

### 3.6 Validating the engine - gradient checking

An analytic gradient is trusted only against an independent oracle. The
central difference

$$
\frac{\partial L}{\partial x_i} \;\approx\;
\frac{L(x + \varepsilon e_i) - L(x - \varepsilon e_i)}{2\varepsilon}
$$

has error $O(\varepsilon^2)$ by Taylor expansion, since the even-order terms
cancel. With $\varepsilon = 10^{-6}$ and float64 arithmetic the truncation
and round-off errors are both far below the comparison tolerance, so
disagreement means a wrong VJP, not noise. The test suite runs this check on
every operation and on composite expressions, including a full
two-layer-network loss differentiated with respect to a weight matrix.

In [4]:
def numerical_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        orig = x[i]
        x[i] = orig + eps
        plus = f(x)
        x[i] = orig - eps
        minus = f(x)
        x[i] = orig
        g[i] = (plus - minus) / (2 * eps)
        it.iternext()
    return g


rng = np.random.default_rng(0)
W_data = rng.standard_normal((4, 3))
x_data = rng.standard_normal((8, 4))

# analytic gradient via spectra
W = Tensor(W_data.copy(), requires_grad=True)
((Tensor(x_data) @ W).relu()).mean().backward()


# numerical gradient via central differences
def loss(w):
    return float(((Tensor(x_data) @ Tensor(w)).relu()).mean().data)


num = numerical_grad(loss, W_data.copy())

print("max abs difference:", np.abs(W.grad - num).max())
assert np.allclose(W.grad, num, atol=1e-8)

max abs difference: 1.647616765243498e-11


*Sections 4 and beyond - initialization theory, optimization, the MNIST
experiment, and the spectral analysis of training - continue in the following
parts of this notebook as the corresponding experiments land in the
repository.*